# Классификация: превышает ли значение SI значение 8

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, f1_score

import tqdm
from tqdm.auto import tqdm

import pickle

In [2]:
RANDOM_STATE = 22

Загружаем данные. Признаки уже стандартизированы, а таргет логарифмирован.

In [3]:
train_df = pd.read_csv("../../data/processed/si8_train.csv")
test_df = pd.read_csv("../../data/processed/si8_test.csv")

In [4]:
X_train = train_df.drop(columns=["SI"])
y_train = train_df["SI"]

X_test = test_df.drop(columns=["SI"])
y_test = test_df["SI"]

print(f"В обучающей выборке содержится {X_train.shape[0]} строк и {X_train.shape[1]} признаков")
print(f"В тестовой выборке содержится {X_test.shape[0]} строк и {X_test.shape[1]} признаков")

В обучающей выборке содержится 798 строк и 30 признаков
В тестовой выборке содержится 200 строк и 30 признаков


Зададим параметры сетки для подбора параметров трех разных моделей регрессии.

In [5]:
param_distributions = {
    "SVC": {
        "model": SVC(probability=True, random_state=42),
        "params": {
            "C": [0.1, 1.0, 10.0],
            "gamma": ["scale", "auto", 0.01]
        }
    },
    "RandomForest": {
        'model': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        'params': {
            'n_estimators': [100, 150, 200],
            'max_depth': [3, 5, 8, 12],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    "LightGBM": {
        "model": LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
        "params": {"learning_rate": [0.01, 0.1], "n_estimators": [100, 200]}
    }
}

In [6]:
best_models = {}
for name, config in tqdm(param_distributions.items()):
    print(f"\n  Модель: {name}")
    print("="*50)
    
    grid_search = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        cv=5,
        scoring="roc_auc",
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    
    # Оцениваем лучшую модель на тестовой выборке
    best_model = grid_search.best_estimator_
    best_models[name] = best_model
    
    preds = best_model.predict(X_test)
    probs = best_model.predict_proba(X_test)[:, 1]

    print(f"    Лучшие параметры: {grid_search.best_params_}")
    print(f"    F1 на тестовой выборке: {f1_score(y_test, preds):.4f}")
    print(f"    ROC-AUC на тестовой выборке: {roc_auc_score(y_test, probs):.4f}")

  0%|          | 0/3 [00:00<?, ?it/s]


  Модель: SVC
    Лучшие параметры: {'C': 10.0, 'gamma': 'auto'}
    F1 на тестовой выборке: 0.5672
    ROC-AUC на тестовой выборке: 0.7119

  Модель: RandomForest
    Лучшие параметры: {'max_depth': 8, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}
    F1 на тестовой выборке: 0.5546
    ROC-AUC на тестовой выборке: 0.7162

  Модель: LightGBM
    Лучшие параметры: {'learning_rate': 0.01, 'n_estimators': 200}
    F1 на тестовой выборке: 0.5833
    ROC-AUC на тестовой выборке: 0.7214


Лучшей моделью по метрике ROC-AUC на тестовой выборке (0.7214) показал себя LightGBM с параметрами 

{'learning_rate': 0.01, 'n_estimators': 200}

Сериализуем лучшую модель и сохраним в файл.

In [7]:
with open('../../models/model_si8_classification.pkl', 'wb') as output:
    pickle.dump(best_models["LightGBM"], output)